Thanks [Yiheng Wang](https://www.kaggle.com/competitions/uw-madison-gi-tract-image-segmentation/discussion/325646)


## Load Libs

In [4]:
import sys
from pathlib import Path

ROOT = Path('.').resolve()
sys.path.append(str(ROOT / 'src'))

DATA_ROOT = ROOT / 'input' / 'uw-madison-gi-tract-image-segmentation'
TRAIN_DIR = DATA_ROOT / 'train'
TRAIN_CSV = DATA_ROOT / 'train.csv'
VAL_CASE_DAYS_CSV = DATA_ROOT / 'splits' / 'val_case_days.csv'

WEIGHTS_FOLD0 = ROOT / 'kaggle_weights' / 'uwmadison-gi-tract-image-segmentation-weights' / 'best_weights_fold_0.pth'


In [5]:
import gc
from glob import glob
import os
import numpy as np
import pandas as pd
import torch
from torch import nn
from monai.inferers import sliding_window_inference
from monai.data import decollate_batch
from monai.handlers.utils import from_engine
from monai.networks.nets import UNet
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm
import json

In [6]:
from monai.data import CacheDataset, DataLoader
from monai.transforms import Compose, Activations, AsDiscrete


## Prepare meta info.

### Thanks awsaf49, this section refers to:
https://www.kaggle.com/code/awsaf49/uwmgi-2-5d-infer-pytorch

In [7]:
from uwgi.constants import CLASSES
from uwgi.data_utils import build_case_day_slices, build_rle_index, load_case_days
from uwgi.datasets import UWGI3DFullVolumeDataset
from uwgi.metrics import dice_per_class_2d_kaggle, hausdorff_score_3d_scipy_kaggle


In [8]:
# Load validation case_day list
val_case_days = load_case_days(str(VAL_CASE_DAYS_CSV))
print('val_case_days:', len(val_case_days))
print('first 5:', val_case_days[:5])


val_case_days: 58
first 5: ['case122_day0', 'case122_day18', 'case122_day24', 'case122_day25', 'case122_day27']


In [9]:
# Build slice path index + RLE index from train.csv
case_day_slices_all = build_case_day_slices(str(TRAIN_DIR))
missing = [cd for cd in val_case_days if cd not in case_day_slices_all]
if missing:
    print('[WARN] missing case_days in TRAIN_DIR:', len(missing))
    # Keep only case_days that exist on disk.
    val_case_days = [cd for cd in val_case_days if cd in case_day_slices_all]

case_day_slices = {cd: case_day_slices_all[cd] for cd in val_case_days}
rle_index = build_rle_index(str(TRAIN_CSV))

print('case_day_slices (val):', len(case_day_slices))


case_day_slices (val): 58


## Produce 3d data list for MONAI DataSet

In [10]:
# Dataset / loader (full volumes with GT masks)
NUM_WORKERS = 0  # set >0 if your notebook environment supports it

val_ds = UWGI3DFullVolumeDataset(
    case_days=val_case_days,
    case_day_slices=case_day_slices,
    rle_index=rle_index,
    cache_in_ram=False,
    verbose_shape_fix=False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print('val volumes:', len(val_ds))


val volumes: 58


In [11]:
# Quick sanity-check: one sample shapes
x0, y0, cd0 = val_ds[0]
print(cd0, 'x', tuple(x0.shape), 'y', tuple(y0.shape), 'classes', list(CLASSES))


case122_day0 x (1, 144, 310, 360) y (3, 144, 310, 360) classes ['large_bowel', 'small_bowel', 'stomach']


## Prepare Transforms, Dataset, DataLoader

In [12]:
class cfg:
    roi_size = (224, 224, 80)
    in_channels = 1
    out_channels = 3
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    weights = [str(WEIGHTS_FOLD0)]  # add more paths to ensemble
    batch_size = 1
    sw_batch_size = 4
    overlap = 0.25
    threshold = 0.5
    normalize = 'volume_max'  # match reference: x/x.max() per volume
    use_tta = True  # flip TTA like the reference notebook


In [13]:
post_pred = Compose([
    Activations(sigmoid=True),
    AsDiscrete(threshold=cfg.threshold),
])


## Prepare Network

In [14]:
# Reference UNet definition (from the original notebook).
model = UNet(
    spatial_dims=3,
    in_channels=cfg.in_channels,
    out_channels=cfg.out_channels,
    channels=(32, 64, 128, 256, 512),
    strides=(2, 2, 2, 2),
    kernel_size=3,
    up_kernel_size=3,
    num_res_units=2,
    act='PRELU',
    norm='BATCH',
    dropout=0.2,
    bias=True,
).to(cfg.device)

# Load the first weight by default (kept as a list to allow ensembling).
ckpt0 = torch.load(cfg.weights[0], map_location='cpu')
state0 = ckpt0.get('model', ckpt0)
model.load_state_dict(state0)
model.eval()


UNet(
  (model): Sequential(
    (0): ResidualUnit(
      (conv): Sequential(
        (unit0): Convolution(
          (conv): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.2, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
        (unit1): Convolution(
          (conv): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.2, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
      )
      (residual): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
    )
    (1): SkipConnection(
      (submodule): Sequential(
        (0): ResidualUnit(
          (conv): Sequential(


## Infer

In [15]:
def infer_mask_cd_hw(
    x_bcdhw: torch.Tensor,
    *,
    model: torch.nn.Module,
    weights: list[str],
    roi_size=(224, 224, 80),
    sw_batch_size=4,
    overlap=0.25,
    threshold=0.5,
    normalize='volume_max',
    tta=True,
) -> np.ndarray:
    """
    x_bcdhw: (B,1,D,H,W) float32 in [0,1]
    returns pred mask: (C,D,H,W) uint8 in {0,1}

    This matches the original notebook's axis order by feeding the model (B,1,H,W,D).
    """
    # (B,1,D,H,W) -> (B,1,H,W,D)
    x = x_bcdhw.permute(0, 1, 3, 4, 2)

    if (normalize or 'none').lower() == 'volume_max':
        mx = x.amax(dim=(2, 3, 4), keepdim=True).clamp_min(1e-6)
        x = x / mx

    dims_list = [()]
    if bool(tta):
        dims_list += [(2,), (3,), (2, 3)]

    pred_all = []
    for w_path in weights:
        ckpt = torch.load(w_path, map_location='cpu')
        state = ckpt.get('model', ckpt)
        model.load_state_dict(state)

        for dims in dims_list:
            x_f = torch.flip(x, dims=dims) if dims else x
            logits = sliding_window_inference(
                x_f.to(cfg.device),
                roi_size,
                int(sw_batch_size),
                model,
                overlap=float(overlap),
            )
            if dims:
                logits = torch.flip(logits, dims=dims)
            pred_all.append(logits)

    logits = torch.mean(torch.stack(pred_all), dim=0)  # (B,C,H,W,D)
    logits = logits.permute(0, 1, 4, 2, 3)  # (B,C,D,H,W)
    prob = torch.sigmoid(logits)
    pred = (prob > float(threshold)).to(torch.uint8)[0].cpu().numpy()
    return pred


In [16]:
# Evaluate on val volumes + compute Kaggle-style score
from pathlib import Path

results = []

dice_sum_all = 0.0
dice_count_all = 0
dice_sum_nonempty = 0.0
dice_count_nonempty = 0
empty_empty_pairs = 0

hd_sum = 0.0
hd_count = 0

model.eval()
torch.set_grad_enabled(False)

for x, y, case_day in tqdm(val_loader, total=len(val_loader)):
    # x:(B,1,D,H,W) y:(B,C,D,H,W)
    x = x.to(cfg.device)
    y_np = (y[0].cpu().numpy() > 0.5).astype(np.uint8)

    pred_np = infer_mask_cd_hw(
        x,
        model=model,
        weights=cfg.weights,
        roi_size=cfg.roi_size,
        sw_batch_size=cfg.sw_batch_size,
        overlap=cfg.overlap,
        threshold=cfg.threshold,
        normalize=cfg.normalize,
        tta=cfg.use_tta,
    )

    c, d, _, _ = pred_np.shape

    # Dice: mean over (slice, class) pairs, empty-empty => 0
    case_dice_sum = 0.0
    case_dice_count = 0
    case_dice_sum_nonempty = 0.0
    case_dice_count_nonempty = 0
    case_empty_empty = 0

    for z in range(d):
        pred2d = pred_np[:, z]
        gt2d = y_np[:, z]

        dice_c = dice_per_class_2d_kaggle(pred2d, gt2d)
        case_dice_sum += float(dice_c.sum())
        case_dice_count += int(dice_c.size)

        denom = pred2d.sum(axis=(1, 2)) + gt2d.sum(axis=(1, 2))
        nonempty = denom > 0
        case_empty_empty += int((~nonempty).sum())
        if bool(nonempty.any()):
            case_dice_sum_nonempty += float(dice_c[nonempty].sum())
            case_dice_count_nonempty += int(nonempty.sum())

    case_dice = case_dice_sum / max(1, case_dice_count)
    case_dice_nonempty = case_dice_sum_nonempty / max(1, case_dice_count_nonempty)

    dice_sum_all += case_dice_sum
    dice_count_all += case_dice_count
    dice_sum_nonempty += case_dice_sum_nonempty
    dice_count_nonempty += case_dice_count_nonempty
    empty_empty_pairs += case_empty_empty

    # Hausdorff (3D) score: mean over classes, empty-empty => 0 by default
    hd_per_class, hd_mean = hausdorff_score_3d_scipy_kaggle(pred_np, y_np, empty_score=0.0)
    hd_sum += float(hd_mean) * int(hd_per_class.size)
    hd_count += int(hd_per_class.size)

    final_score = 0.4 * float(case_dice) + 0.6 * float(hd_mean)
    final_score_nonempty = 0.4 * float(case_dice_nonempty) + 0.6 * float(hd_mean)

    results.append({
        'case_day': str(case_day[0]),
        'dice': float(case_dice),
        'dice_nonempty': float(case_dice_nonempty),
        'hausdorff_score': float(hd_mean),
        'final_score': float(final_score),
        'final_score_nonempty': float(final_score_nonempty),
        'count_total': int(case_dice_count),
        'count_nonempty': int(case_dice_count_nonempty),
        'empty_empty_pairs': int(case_empty_empty),
    })

# Global means
mean_dice = float(dice_sum_all / max(1, dice_count_all))
mean_dice_nonempty = float(dice_sum_nonempty / max(1, dice_count_nonempty))
mean_hausdorff_score = float(hd_sum / max(1, hd_count))
final_score = 0.4 * mean_dice + 0.6 * mean_hausdorff_score
final_score_nonempty = 0.4 * mean_dice_nonempty + 0.6 * mean_hausdorff_score

print('mean_dice:', mean_dice)
print('mean_dice_nonempty:', mean_dice_nonempty)
print('mean_hausdorff_score:', mean_hausdorff_score)
print('final_score:', final_score)
print('final_score_nonempty:', final_score_nonempty)

run_tag = f"monai_nb_val_reference_unet_norm_{cfg.normalize}_tta_{'flip' if cfg.use_tta else 'none'}_{Path(cfg.weights[0]).name}"
out_dir = Path('outputs') / 'eval_runs' / run_tag
out_dir.mkdir(parents=True, exist_ok=True)

df = pd.DataFrame(results).sort_values('case_day')
df.to_csv(out_dir / 'eval_per_case.csv', index=False)

summary = {
    'model_name': 'reference_unet',
    'weights': cfg.weights,
    'roi_size': list(cfg.roi_size),
    'sw_batch_size': int(cfg.sw_batch_size),
    'overlap': float(cfg.overlap),
    'threshold': float(cfg.threshold),
    'normalize': str(cfg.normalize),
    'tta': bool(cfg.use_tta),
    'mean_dice': mean_dice,
    'mean_dice_nonempty': mean_dice_nonempty,
    'mean_hausdorff_score': mean_hausdorff_score,
    'final_score': float(final_score),
    'final_score_nonempty': float(final_score_nonempty),
    'dice_count_all': int(dice_count_all),
    'dice_count_nonempty': int(dice_count_nonempty),
    'empty_empty_pairs': int(empty_empty_pairs),
    'num_volumes': int(len(results)),
}
(out_dir / 'eval_summary.json').write_text(json.dumps(summary, indent=2))

print('wrote:', out_dir)


  0%|          | 0/58 [00:00<?, ?it/s]/home/jeremiah/miniconda/envs/segmentation/lib/python3.10/site-packages/monai/inferers/utils.py:226: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
/home/jeremiah/miniconda/envs/segmentation/lib/python3.10/site-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different

mean_dice: 0.11956131356577514
mean_dice_nonempty: 0.3807829512616156
mean_hausdorff_score: 0.5391848007666653
final_score: 0.3713354058863092
final_score_nonempty: 0.4758240609646454
wrote: outputs/eval_runs/monai_nb_val_reference_unet_norm_volume_max_tta_flip_best_weights_fold_0.pth


In [17]:
df.head()


,case_day,dice,dice_nonempty,hausdorff_score,final_score,final_score_nonempty,count_total,count_nonempty,empty_empty_pairs
0,case122_day0,0.266732,0.768188,0.935683,0.668103,0.868685,432,150,282
1,case122_day18,0.315794,0.836952,0.948427,0.695374,0.903837,432,163,269
2,case122_day24,0.295801,0.803686,0.953698,0.690539,0.893693,432,159,273
3,case122_day25,0.268997,0.730860,0.953785,0.679870,0.864615,432,159,273
4,case122_day27,0.286888,0.779470,0.942598,0.680314,0.877347,432,159,273


In [18]:
summary


{'model_name': 'reference_unet',
 'weights': ['/home/jeremiah/github/UW-Madison_GI_Tract_Image_Segmentation/kaggle_weights/uwmadison-gi-tract-image-segmentation-weights/best_weights_fold_0.pth'],
 'roi_size': [224, 224, 80],
 'sw_batch_size': 4,
 'overlap': 0.25,
 'threshold': 0.5,
 'normalize': 'volume_max',
 'tta': True,
 'mean_dice': 0.11956131356577514,
 'mean_dice_nonempty': 0.3807829512616156,
 'mean_hausdorff_score': 0.5391848007666653,
 'final_score': 0.3713354058863092,
 'final_score_nonempty': 0.4758240609646454,
 'dice_count_all': 24864,
 'dice_count_nonempty': 7807,
 'empty_empty_pairs': 17057,
 'num_volumes': 58}